# HF model supply chain paper — robustness variant runs (Colab share)

The work is split with the laptop (local). **This notebook runs 7 variants**:
`keeptv, noquant, coarse, strictlang, other2unk, strictcom, noov`
(the laptop runs `main, tierT0, tierT2, declared`)

- Results are saved under `04_results/tables/2026-09-25_full_<variant>/` in the project folder on Google Drive and are shared with the laptop.
- One variant needs about 3 GB of memory, so free Colab (about 12 GB) runs **2 at a time**.
- Expected time: 40-60 min per variant × 7 ÷ 2 ≈ **3-4 hours**. Keep the browser tab open (closing it may drop the session).
- A dropped session is fine: on rerun, **finished variants are skipped**.

Order: run the cells from the top, one at a time (Shift+Enter).

In [1]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2) Change to the project folder, check memory
import os, subprocess
ROOT = '/content/drive/MyDrive/논문_AiModels'
os.chdir(ROOT + '/03_code')
print(os.getcwd())
!free -g | head -2
!nproc

/content/drive/MyDrive/논문_AiModels/03_code
               total        used        free      shared  buff/cache   available
Mem:              12           1           8           0           3          11
2


In [3]:
# 3) Version check: has the latest code and data fixed on the laptop reached Drive? (if any line says FAIL, stop and retry a little later)
import pandas as pd
checks = {
    'runner supports --workers': '--workers' in open('run_all_experiments.py', encoding='utf-8').read(),
    'sim v2 (availability shock)': 'availability' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'sim v2 (union-matched null)': 'def union_matched' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'commercial fix (#4)': '"responsible_ai", "vendor_custom"}' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'tests v2 present': 'test_availability_semantics_and_mirror' in open('03_simulation/test_removal_sim.py', encoding='utf-8').read(),
}
f = pd.read_parquet(ROOT + '/02_data/processed/2026-09-25/model_flags.parquet', columns=['in_T1'])
checks['cleaning fix #1 + reconnection (T1 = 1,631,600)'] = int(f['in_T1'].sum()) == 1631600
e = pd.read_parquet(ROOT + '/02_data/processed/2026-09-25/edges_all.parquet', columns=['parent_resolution', 'temporal_ok'])
checks['parent reconnection (5,820 edges)'] = int(e['parent_resolution'].notna().sum()) == 5820
checks['reconnected edges not time-judged (fix 12:50)'] = int(e.loc[e['parent_resolution'].notna(), 'temporal_ok'].isna().sum()) == 5820
checks['degradation metric'] = 'def degradation' in open('03_simulation/removal_sim.py', encoding='utf-8').read()
checks['arch lower bound'] = os.path.exists(ROOT + '/02_data/processed/2026-09-25/arch.parquet')
for k, v in checks.items():
    print(('OK  ' if v else 'FAIL') + '  ' + k)
assert all(checks.values()), 'Drive sync has not finished yet. Run this cell again in a few minutes.'

OK    runner supports --workers
OK    sim v2 (availability shock)
OK    sim v2 (union-matched null)
OK    commercial fix (#4)
OK    tests v2 present
OK    cleaning fix #1 + reconnection (T1 = 1,631,600)
OK    parent reconnection (5,820 edges)
OK    reconnected edges not time-judged (fix 12:50)
OK    degradation metric
OK    arch lower bound


In [4]:
# 4) Required packages (most are preinstalled on Colab) + unit tests
!pip -q install -r requirements.txt
!cd 03_simulation && python test_removal_sim.py
!cd 02_graph && python test_rules.py
!python 04_analysis/hypothesis_tests.py --selftest

all tests passed
all rule tests passed
selftest passed: {'H2_availability': {'k_significant': 0, 'k_total': 3, 'supported': False}, 'H2_legal': {'k_significant': 3, 'k_total': 3, 'supported': True}}


In [5]:
# 5) Run the 7 variants (2 at a time). Takes a few hours. Check progress in cell 6) below
!python -u run_all_experiments.py ../02_data/processed/2026-09-25 \
    --only keeptv noquant coarse strictlang other2unk strictcom noov \
    --workers 2 --host colab

[21:30:42] host=colab workers=2 variants=['keeptv', 'noquant', 'coarse', 'strictlang', 'other2unk', 'strictcom', 'noov']
[21:30:42] skip keeptv (done)
[21:30:42] skip noquant (done)
[21:30:42] skip coarse (done)
[21:30:42] skip strictlang (done)
[21:30:42] skip other2unk (done)
[21:30:42] skip strictcom (done)
[21:30:42] skip noov (done)
all done


### Progress check (cannot run while cell 5 is running; use it after reconnecting from a dropped session)

In [6]:
# 6) Progress: last log line and completion status per variant
import glob, os
for name in ['keeptv', 'noquant', 'coarse', 'strictlang', 'other2unk', 'strictcom', 'noov']:
    done = os.path.exists(f'{ROOT}/04_results/tables/2026-09-25_full_{name}/hypothesis_verdicts.json')
    logs = glob.glob(f'{ROOT}/04_results/logs/2026-09-25_{name}_colab.log')
    last = open(logs[0], encoding='utf-8').read().strip().splitlines()[-1][:110] if logs else '(not started)'
    print(f"{'DONE' if done else '....'}  {name:11s} {last}")

DONE  keeptv      [14:39:12] exit 0 (1s)
DONE  noquant     [07:56:16] exit 0 (1s)
DONE  coarse      [14:37:40] exit 0 (1s)
DONE  strictlang  [16:59:54] exit 0 (1s)
DONE  other2unk   [17:02:15] exit 0 (1s)
DONE  strictcom   [19:23:35] exit 0 (3s)
DONE  noov        [19:25:05] exit 0 (1s)


In [ ]:
# 7) 2026-09-27 review response: run the matched-null criterion sensitivities (tol90, tol99) at the same time. Rerun after a drop to resume
%cd /content/drive/MyDrive/논문_AiModels/03_code
!ls run_logged.py colab_jobs.py
!python colab_jobs.py